In [3]:
import numpy as np
import pandas as pd
from glob import glob
import networkx as nx
import matplotlib.pyplot as plt
import ipycytoscape

from pyvis.network import Network
from IPython.display import IFrame
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

DATA_RAW_FILE = 'DATA_RAW_CLEANED_20250401_20251231_ANTOFAGASTA_8834.parquet'

CLEANING_DATA = False
SAMPLING = '3h'

DATA_RAW_FILE = [x for x in glob('../**', recursive=True) if DATA_RAW_FILE in x][0]
DATA_RAW_FILE

'..\\DataCleaning\\DATA_RAW_CLEANED_20250401_20251231_ANTOFAGASTA_8834.parquet'

## READING DATA RAW

### CLEANING FUNCTIONS

In [4]:
def replace_outliers_with_nan_kmeans(series, n_clusters=3, contamination=0.05, random_state=42):

    if not isinstance(series, pd.Series):
        raise TypeError("Input must be a pandas Series.")

    valid_data = series.dropna()
    
    if len(valid_data) == 0:
        return series 
        
    X = valid_data.values.reshape(-1, 1)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X)

    all_distances = kmeans.transform(X)
    dist_to_center = np.min(all_distances, axis=1)

    threshold = np.quantile(dist_to_center, 1 - contamination)
    outlier_mask_valid = dist_to_center > threshold
    outlier_indices = valid_data.index[outlier_mask_valid]

    series_clean = series.copy()
    
    if not pd.api.types.is_float_dtype(series_clean):
        series_clean = series_clean.astype(float)
        
    series_clean.loc[outlier_indices] = np.nan

    return series_clean

def fix_missing_points(series, limit=4):
    return series.interpolate(method='linear', limit=limit, limit_direction='forward')

def fill_nan_with_time_mean(series):

    if not isinstance(series, pd.Series):
        raise TypeError("Input must be a pandas Series.")
        
    filled_series = series.copy()

    if not isinstance(filled_series.index, pd.DatetimeIndex):
        try:
            filled_series.index = pd.to_datetime(filled_series.index)
        except Exception as e:
            raise ValueError("Series index must be convertible to DatetimeIndex.") from e

    time_means = filled_series.groupby([filled_series.index.hour, filled_series.index.minute]).transform('mean')

    filled_series = filled_series.fillna(time_means)

    return filled_series





In [5]:
DATA_RAW = pd.read_parquet(DATA_RAW_FILE)

if CLEANING_DATA:
    ## CONVERTING TO 1 HOUR SAMPLE


    print("Aplying outlier replacement")
    DATA_RAW = DATA_RAW.apply(lambda x: replace_outliers_with_nan_kmeans(x))

    print("Aplying fix missing points")
    DATA_RAW = DATA_RAW.apply(lambda x: fix_missing_points(x))

    print("Aplying fill nan with time mean")
    DATA_RAW = DATA_RAW.apply(lambda x: fill_nan_with_time_mean(x))

    DATA_RAW = DATA_RAW.T

DATA_RAW = DATA_RAW.T.resample(SAMPLING).max().T
DATA_RAW = DATA_RAW[DATA_RAW.mean(axis=1) > 1]

#Replacing zeros with values 
DATA_RAW = DATA_RAW.mask(DATA_RAW < 1)     #.replace(0, np.nan)
DATA_RAW = DATA_RAW.T.apply(fill_nan_with_time_mean).T
#DATA_RAW = DATA_RAW.round(1)


NODES = pd.DataFrame( list(set(DATA_RAW.index.get_level_values(0).tolist() + DATA_RAW.index.get_level_values(1).tolist())) ).rename(columns={0: 'name'})
EDGES_ID = pd.DataFrame([x for x in range(1000, 1000+len(DATA_RAW))], index=DATA_RAW.index, columns=['ID'])
EDGES_I_INV = {x['ID']:(x['A'],x['B']) for x in EDGES_ID.reset_index().to_dict('records')}


DATA_RAW.sample(10)

,,2025-04-01 00:00:00,2025-04-01 03:00:00,2025-04-01 06:00:00,2025-04-01 09:00:00,2025-04-01 12:00:00,2025-04-01 15:00:00,2025-04-01 18:00:00,2025-04-01 21:00:00,2025-04-02 00:00:00,2025-04-02 03:00:00,...,2025-12-30 18:00:00,2025-12-30 21:00:00,2025-12-31 00:00:00,2025-12-31 03:00:00,2025-12-31 06:00:00,2025-12-31 09:00:00,2025-12-31 12:00:00,2025-12-31 15:00:00,2025-12-31 18:00:00,2025-12-31 21:00:00
A,B,,,,,,,,,,,,,,,,,,,,,
950d_antofagasta_mr_a_2001,910c_aguas_el_salto_csr_r8_02411,637.955695,185.729042,291.895793,417.240424,461.790142,568.340867,738.078767,707.771337,651.106299,210.186708,...,566.561665,665.061890,652.949557,325.611208,282.059045,357.546804,397.456541,578.650064,561.433750,758.052808
950d_la_negra_preagg,950d_antofagasta_corp_a_02001,57.298094,180.819119,479.115286,517.948958,564.053393,506.469052,391.684490,107.039138,71.622617,180.819119,...,366.628452,270.554611,362.145482,200.847073,241.615657,600.428772,630.341707,340.216091,271.258294,347.973622
950d_antofagasta_mr_a_2001,910c_tocopilla_norte_csr_02201,297.629400,69.547270,232.381763,251.359031,260.808106,273.484498,255.631261,416.433376,335.273227,86.698219,...,474.965438,537.970801,509.663581,240.125413,246.596241,381.568959,399.219629,453.333025,473.141246,487.370432
sara_anf_av_bllvst_r3_csr_02611,950d_antofagasta_mr_a_2001,101.485466,153.829506,269.190756,325.898626,401.601388,378.162530,355.885946,170.343385,126.843447,153.829506,...,113.117478,376.157691,69.708738,153.829506,47.178852,3.314051,401.601388,105.161527,113.105136,376.157691
910c_anto_corvalis_csr_r5_02052,02_052,157.806503,58.992390,131.417519,132.274197,133.023203,157.778201,181.335825,162.077299,156.048227,70.836874,...,140.135571,155.485140,156.201324,100.139484,83.529204,94.543624,130.640372,142.681112,147.254919,143.159725
910d_segund_gomez_csr_corp_02004,02_004,174.505968,66.761953,206.686748,297.302100,310.826146,302.976900,273.262950,211.474089,190.278002,73.021799,...,289.221178,236.052519,202.485702,105.713100,168.084167,272.551602,297.265037,283.457772,247.025974,157.956777
910c_aguas_el_salto_csr_r8_02411,02_411,164.590546,41.474426,87.617015,86.432865,113.699474,147.115283,204.902948,197.892483,167.942511,42.289267,...,187.535296,218.136580,189.845908,73.880834,78.543229,117.868110,129.520107,167.504694,162.490667,164.034877
910c_aguas_anf_prat_csr_r5_02404,910c_aguas_anf_balm_csr_r5_02413,217.001487,100.191051,134.883082,159.296828,170.948137,197.515527,198.014098,261.729383,233.257681,91.737292,...,241.663555,258.438173,253.438237,197.509732,149.601009,185.006336,203.371129,224.245904,226.243222,244.267860
950d_antofagasta_corp_a_02001,950d_mall_copiapo_preagg,39.068630,14.660161,30.251037,34.743651,45.422071,89.501608,49.763833,58.678398,45.191147,18.014903,...,146.223208,67.397448,67.563966,62.364926,72.807056,118.418384,164.859096,118.041902,93.023618,45.226330


In [6]:
def draw_nodes_pyvis(data_raw: pd.DataFrame, filename: str = 'graph.html', directed: bool = True, sample_time: bool = True):
    # 1. Prepare Data (Same logic as before)
    if not sample_time:
        #data = data_raw.mean(axis=1).to_frame().round(0).rename(columns={0: 'bw'})
        data = data_raw.loc[:,data_raw.sum().idxmax()].to_frame().round(0).rename(columns={0: 'bw'})
    else:
        _sample_date =  data_raw.T.sample(1).index.values[0]
        data = data_raw[_sample_date].to_frame().round(0).rename(columns={_sample_date: 'bw'})
    
    # Node Data
    nodes_list = list(set(data.index.get_level_values(0).astype(str).tolist() + 
                          data.index.get_level_values(1).astype(str).tolist()))
    nodes_data = pd.DataFrame(nodes_list, columns=["id"])
    
    # Assign Colors/Sizes
    nodes_data['color'] = 'blue'
    nodes_data['size'] = 15  # Pyvis uses different scaling, smaller is better
    
    # Red Nodes
    hr_mask = nodes_data['id'].str.match(r'.*hr.*')
    nodes_data.loc[hr_mask, 'color'] = 'red'
    nodes_data.loc[hr_mask, 'size'] = 30

    # Yellow Nodes
    mr_mask = nodes_data['id'].str.match(r'.*mr_.*')
    nodes_data.loc[mr_mask, 'color'] = '#FFD700' # Hex for yellow often works better
    nodes_data.loc[mr_mask, 'size'] = 25
    
    # Green Nodes
    id_mask = nodes_data['id'].str.match(r'\d+[isp]?_\d+$')
    nodes_data.loc[id_mask, 'color'] = 'green'
    nodes_data.loc[id_mask, 'size'] = 10

    nodes_dict = nodes_data.to_dict('records')
    
    # Edge Data
    edge_df = data.reset_index()
    edge_df.iloc[:, 0] = edge_df.iloc[:, 0].astype(str)
    edge_df.iloc[:, 1] = edge_df.iloc[:, 1].astype(str)
    edge_data = list(edge_df.itertuples(index=False, name=None))

    # 2. Build Network
    # cdn_resources='remote' ensures it works without local JS files
    net = Network(height='600px', width='100%', notebook=True, cdn_resources='remote', directed=directed) 
                  
    
    for node in nodes_dict:
        net.add_node(node['id'], color=node['color'], size=node['size'], label=node['id'])

    for src, tgt, bw in edge_data:
        net.add_edge(
            src, 
            tgt, 
            title=f"Bandwidth: {bw}",  # Shows on hover
            label=str(int(bw)),             # Shows permanently on the line
            width=2,
            font={'align': 'middle', 'strokeWidth': 0, 'background': 'white'} # Makes text easier to read
        )

    # 3. Physics Options (Similar to 'cose')
    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08)
    
    # 4. Save and Show
    # This writes a file 'graph.html' to your folder. Open this file to see the graph!
    net.show(filename) 
    return f"Graph saved to '{filename}'. Open this file to view."


draw_nodes_pyvis(DATA_RAW, filename='graph_year.html', sample_time=True)


graph_year.html


"Graph saved to 'graph_year.html'. Open this file to view."


## GMAN - Graph Neural Networks for Time Series Forecasting

### Time Series Forecasting -  Raw Data

In [47]:
order_list_nodes = list(set(list(DATA_RAW.index.get_level_values(0)) + list(DATA_RAW.index.get_level_values(1))))
order_list_nodes.sort()
order_list_nodes = {order_list_nodes[i]: 1000+i for i in range(len(order_list_nodes))}
order_list_nodes

{'02_003': 1000,
 '02_004': 1001,
 '02_005': 1002,
 '02_006': 1003,
 '02_009': 1004,
 '02_016': 1005,
 '02_020': 1006,
 '02_030': 1007,
 '02_045': 1008,
 '02_046': 1009,
 '02_051': 1010,
 '02_052': 1011,
 '02_066': 1012,
 '02_075': 1013,
 '02_102': 1014,
 '02_141': 1015,
 '02_158': 1016,
 '02_208': 1017,
 '02_301': 1018,
 '02_302': 1019,
 '02_303': 1020,
 '02_304': 1021,
 '02_312': 1022,
 '02_323': 1023,
 '02_350': 1024,
 '02_361': 1025,
 '02_404': 1026,
 '02_405': 1027,
 '02_406': 1028,
 '02_407': 1029,
 '02_408': 1030,
 '02_410': 1031,
 '02_411': 1032,
 '02_413': 1033,
 '02_420': 1034,
 '02_421': 1035,
 '02_422': 1036,
 '02_423': 1037,
 '02_469': 1038,
 '02_474': 1039,
 '02_489': 1040,
 '02_521': 1041,
 '02_527': 1042,
 '02_541': 1043,
 '02_605': 1044,
 '02_607': 1045,
 '02_611': 1046,
 '910c_afs_camar_oriente_r6_csr_02605': 1047,
 '910c_aguas_anf_balm_csr_r5_02413': 1048,
 '910c_aguas_anf_ctro_csr_r2_02408': 1049,
 '910c_aguas_anf_prat_csr_r5_02404': 1050,
 '910c_aguas_el_salto_csr_

In [54]:
DATA_RAW_G = DATA_RAW.copy()
DATA_RAW_G.index = pd.MultiIndex.from_tuples( [(order_list_nodes[x1],order_list_nodes[x2]) for x1,x2 in DATA_RAW_G.index] )
DATA_RAW_G.index.names = ['A','B']
DATA_RAW_G

2025-04-01 00:00:00  2025-04-01 03:00:00  2025-04-01 06:00:00  \
A    B                                                                     
1047 1044           104.169947            50.028195            87.689750   
     1084           490.551674           265.059623           446.529602   
1048 1033           254.247851           139.477457           226.204348   
1049 1030           218.610596            86.827460           153.770096   
     1063            22.276345             9.096051            27.180364   
...                        ...                  ...                  ...   
1105 1104          5142.731016          1701.737813          3936.478878   
1106 1041            33.990899            33.361908            89.404894   
1107 1046            77.786488            18.159881            74.699412   
     1060            48.313964            78.814272           131.183604   
     1097           101.485466           153.829506           269.190756   

           2025-04-01 09:00:00  2025-04-01 12:00:00  2025-04-01 15:00:00  \
A    B                                                                     
1047 1044            95.321859            99.501067           109.306061   
     1084           473.186217           567.984604           618.127702   
1048 1033           235.307673           250.936560           278.153769   
1049 1030           165.460374           166.803838           188.475483   
     1063            30.043914            31.115959            33.431765   
...                        ...                  ...                  ...   
1105 1104          4236.144112          5067.927830          5400.304248   
1106 1041           102.232529           148.197902           142.347374   
1107 1046            85.413109            93.903533            82.652772   
     1060           167.648699           202.164471           188.906830   
     1097           325.898626           401.601388           378.162530   

           2025-04-01 18:00:00  2025-04-01 21:00:00  2025-04-02 00:00:00  \
A    B                                                                     
1047 1044           115.182549           125.449028           111.176075   
     1084           618.785173           601.207634           497.430170   
1048 1033           278.583791           248.445105           241.405377   
1049 1030           208.749833           255.551094           216.615291   
     1063            35.758373            28.584328            21.590419   
...                        ...                  ...                  ...   
1105 1104          6011.147125          6438.459476          5351.323492   
1106 1041            35.693579            44.186534            32.422297   
1107 1046            97.782811           107.765217            83.457660   
     1060           177.501036            84.074808            60.385503   
     1097           355.885946           170.343385           126.843447   

           2025-04-02 03:00:00  ...  2025-12-30 18:00:00  2025-12-30 21:00:00  \
A    B                          ...                                             
1047 1044            47.994109  ...           125.490440           124.395910   
     1084           237.841806  ...           755.172957           764.306148   
1048 1033           121.965148  ...           240.418146           252.622117   
1049 1030            78.165764  ...           237.645295           277.629092   
     1063             9.111149  ...            25.706531            22.630723   
...                        ...  ...                  ...                  ...   
1105 1104          2166.761718  ...          5792.387846          6421.542096   
1106 1041            24.189317  ...            36.706486            26.682791   
1107 1046            24.139841  ...           124.719584           130.185166   
     1060            78.814272  ...            55.779479           187.566305   
     1097           153.829506  ...           113.117478           376.1

In [55]:
DATA_RAW_G.index

MultiIndex([(1047, 1044),
            (1047, 1084),
            (1048, 1033),
            (1049, 1030),
            (1049, 1063),
            (1050, 1026),
            (1050, 1048),
            (1051, 1032),
            (1051, 1053),
            (1052, 1035),
            ...
            (1104, 1099),
            (1104, 1100),
            (1105, 1096),
            (1105, 1098),
            (1105, 1099),
            (1105, 1104),
            (1106, 1041),
            (1107, 1046),
            (1107, 1060),
            (1107, 1097)],
           names=['A', 'B'], length=122)

## ADjacency MAtrix

In [36]:
ADJ_M = DATA_RAW.copy()
ADJ_M

2025-04-01 00:00:00  \
A                                   B                                                      
910c_afs_camar_oriente_r6_csr_02605 02_605                                    104.169947   
                                    910c_rio_lauca_csr_r6_02045               490.551674   
910c_aguas_anf_balm_csr_r5_02413    02_413                                    254.247851   
910c_aguas_anf_ctro_csr_r2_02408    02_408                                    218.610596   
                                    910c_av_arg_toyota_csr_r2_02323            22.276345   
...                                                                                  ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001          5142.731016   
sara_3_brigada_acrzda_02521         02_521                                     33.990899   
sara_anf_av_bllvst_r3_csr_02611     02_611                                     77.786488   
                                    910c_antofa_paihuano_csr_02469             48.313964   
                                    950d_antofagasta_mr_a_2001                101.485466   

                                                                     2025-04-01 03:00:00  \
A                                   B                                                      
910c_afs_camar_oriente_r6_csr_02605 02_605                                     50.028195   
                                    910c_rio_lauca_csr_r6_02045               265.059623   
910c_aguas_anf_balm_csr_r5_02413    02_413                                    139.477457   
910c_aguas_anf_ctro_csr_r2_02408    02_408                                     86.827460   
                                    910c_av_arg_toyota_csr_r2_02323             9.096051   
...                                                                                  ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001          1701.737813   
sara_3_brigada_acrzda_02521         02_521                                     33.361908   
sara_anf_av_bllvst_r3_csr_02611     02_611                                     18.159881   
                                    910c_antofa_paihuano_csr_02469             78.814272   
                                    950d_antofagasta_mr_a_2001                153.829506   

                                                                     2025-04-01 06:00:00  \
A                                   B                                                      
910c_afs_camar_oriente_r6_csr_02605 02_605                                     87.689750   
                                    910c_rio_lauca_csr_r6_02045               446.529602   
910c_aguas_anf_balm_csr_r5_02413    02_413                                    226.204348   
910c_aguas_anf_ctro_csr_r2_02408    02_408                                    153.770096   
                                    910c_av_arg_toyota_csr_r2_02323            27.180364   
...                                                                                  ...   
ne8000m14_antofagasta_hr_b_2001     ne8000m14_antofagasta_hr_a_2001          3936.478878   
sara_3_brigada_acrzda_02521         02_521                                     89.404894   
sara_anf_av_bllvst_r3_csr_02611     02_611                                     74.699412   
                                    910c_antofa_paihuano_csr_02469            131.183604   
                                    950d_antofagasta_mr_a_2001                269.190756   

                                                                     2025-04-01 09:00:00  \
A                                   B                                                      
910c_afs_camar_oriente_r6_csr_02605 02_605                                     95.321859   
                                    910c_rio_lauca_csr_r6_02045               473.186217   
910c_aguas_anf_balm_csr_r5_02413    02_413                                    235.307673   
910c_aguas_anf_ctr

In [26]:
G = nx.DiGraph()
G.add_edges_from(DATA_RAW.index.values)

ADJ_M = nx.to_pandas_adjacency(G, dtype=int)
ADJ_M = ADJ_M.loc[DATA_RAW.index.get_level_values(0), DATA_RAW.index.get_level_values(1)]
ADJ_M.to_numpy()

array([[1, 1, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1]])

In [28]:
adj_matrix = ADJ_M.to_numpy()
rows, cols = np.where(adj_matrix > 0)

G1 = nx.Graph() 
for r, c in zip(rows, cols):
    G1.add_edge(r, c, weight=adj_matrix[r, c])

In [30]:
G1.edges()

EdgeView([(0, 0), (0, 1), (1, 1), (2, 2), (3, 3), (3, 4), (3, 93), (4, 4), (4, 93), (4, 91), (4, 92), (4, 94), (4, 95), (4, 96), (4, 97), (4, 98), (4, 99), (4, 100), (93, 91), (93, 92), (93, 19), (93, 43), (93, 86), (93, 93), (93, 94), (93, 95), (93, 96), (93, 97), (93, 98), (93, 99), (93, 100), (5, 5), (5, 6), (6, 6), (7, 7), (7, 8), (8, 8), (9, 9), (10, 10), (10, 11), (10, 63), (11, 11), (11, 63), (11, 62), (63, 62), (63, 63), (12, 12), (13, 13), (13, 14), (14, 14), (15, 15), (16, 16), (16, 17), (17, 17), (18, 18), (18, 19), (18, 97), (19, 19), (19, 97), (19, 91), (19, 92), (19, 94), (19, 95), (19, 96), (19, 98), (19, 99), (19, 100), (97, 91), (97, 92), (97, 94), (97, 95), (97, 96), (97, 43), (97, 86), (97, 97), (97, 98), (97, 99), (97, 100), (20, 20), (21, 21), (22, 22), (22, 23), (23, 23), (24, 24), (24, 25), (24, 88), (25, 25), (25, 88), (25, 80), (25, 81), (25, 82), (25, 83), (25, 84), (25, 85), (25, 86), (25, 87), (25, 89), (25, 90), (88, 80), (88, 81), (88, 82), (88, 83), (88, 

In [ ]:
DATA_TS = DATA_RAW.copy()
DATA_TS.index = EDGES_ID['ID']
DATA_TS = DATA_TS.T

print(f"Shape of the time series: {DATA_TS.shape}")
DATA_TS.sample(10)

Shape of the time series: (2200, 122)


ID,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121
2025-06-13 18:00:00,119.311805,530.698691,254.589072,242.920334,42.601650,186.040258,276.736424,184.689193,527.826409,99.599433,...,1276.310048,661.831965,831.739679,3668.503078,919.964716,7670.885675,35.243008,114.868873,99.549223,198.638854
2025-07-24 18:00:00,132.860718,858.021868,259.395908,246.400745,44.547175,186.040258,262.201265,211.128156,508.312554,134.858685,...,1.543237,672.857410,878.261133,4785.099878,1035.241298,6760.107342,47.302471,126.030223,177.501036,355.885946
2025-10-18 18:00:00,138.284039,552.574199,242.450476,270.489077,19.781587,235.787125,245.829414,220.142735,567.190542,142.503165,...,672.313053,708.115355,744.162324,5406.211208,918.557890,6985.448250,42.628999,131.796500,173.242858,277.510662
2025-04-21 03:00:00,86.850392,232.841873,170.469442,95.708994,9.652987,85.468895,180.385042,85.395900,117.343925,37.944764,...,550.619687,249.609697,328.053646,1654.337558,277.516279,2275.744858,17.712129,20.786339,78.814272,153.829506
2025-09-10 00:00:00,129.049918,793.312181,228.790959,229.808029,7.382565,158.832570,234.331551,199.556946,262.213604,120.016882,...,618.290076,660.064425,732.305719,3813.812116,1008.632361,6071.086647,52.214403,81.140643,149.353305,305.363889
2025-05-17 12:00:00,124.084965,708.801154,252.005792,218.262671,38.205774,168.298244,253.015373,152.900635,286.640266,92.817965,...,1209.358935,634.596539,759.046809,2282.734577,867.225774,7132.777356,30.955809,91.580722,202.164471,401.601388
2025-11-17 18:00:00,135.635239,271.536987,268.318927,275.689110,18.194040,240.056686,267.032514,217.812484,22.496557,107.553436,...,641.455872,642.272135,739.741280,5335.881890,980.277346,5804.677878,32.602510,149.969396,540.167301,1078.780467
2025-04-04 09:00:00,102.497043,528.081887,226.811301,160.497802,38.722708,142.086312,283.137380,88.360364,236.375522,93.360510,...,403.266161,573.117979,751.687427,4018.375682,668.359721,4513.109595,130.228793,82.539220,167.648699,325.898626
2025-11-01 21:00:00,136.633904,562.463125,262.077757,288.734070,27.672309,241.150706,263.218102,204.877817,549.929224,136.859369,...,631.490883,682.408254,822.539551,4834.807068,1051.168074,6893.560641,43.936722,146.510788,123.004547,290.027028
2025-10-23 03:00:00,35.113107,176.680994,100.503025,125.131899,11.792122,52.989970,96.075324,87.889947,104.929515,25.730156,...,245.887475,249.868796,266.238586,1063.062529,343.554698,1883.108602,15.106110,40.521156,21.876934,39.091728


### SOME STATISTICS

In [ ]:
print(f"The mean number of point by day is {DATA_TS.resample('D').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by week is {DATA_TS.resample('W').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by month is {DATA_TS.resample('ME').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by year is {DATA_TS.resample('YE').count().iloc[:,0].mean().round().astype(int)}")


The mean number of point by day is 8
The mean number of point by week is 55
The mean number of point by month is 244
The mean number of point by year is 2200


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [15]:
NUM_NODES = 50
HIST_STEPS = 12
PRED_STEPS = 12
INPUT_DIM = 1
OUTPUT_DIM = 1
D_MODEL = 64
    
    # 1. Simulate Graph & Node2Vec
adj_matrix = np.random.randint(0, 2, (NUM_NODES, NUM_NODES))
adj_matrix

array([[1, 1, 1, ..., 0, 0, 1],
       [1, 1, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 0, 1, 1],
       ...,
       [0, 0, 0, ..., 1, 1, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 1, 1]])

In [ ]:
# ==========================================
# 1. NODE2VEC PRE-PROCESSING
# ==========================================
def generate_node2vec_embeddings(adj_matrix, embedding_dim=64):
    """
    Generates node embeddings using Node2Vec.
    adj_matrix: (N, N) numpy array (adjacency matrix)
    """
    try:
        import networkx as nx
        from node2vec import Node2Vec
    except ImportError:
        print("Please install node2vec: pip install node2vec networkx")
        # Return random vectors for demonstration purposes if library missing
        return torch.randn(adj_matrix.shape[0], embedding_dim)

    # Create Graph
    rows, cols = np.where(adj_matrix > 0)
    G = nx.Graph() 
    for r, c in zip(rows, cols):
        G.add_edge(r, c, weight=adj_matrix[r, c])
        
    # Run Node2Vec
    # P=1, Q=1 is standard equivalent to DeepWalk, adjust for BFS/DFS bias
    node2vec = Node2Vec(G, dimensions=embedding_dim, walk_length=30, num_walks=200, workers=1, quiet=True)
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    
    # Extract vectors in order
    num_nodes = adj_matrix.shape[0]
    embeddings = np.zeros((num_nodes, embedding_dim))
    for i in range(num_nodes):
        if str(i) in model.wv:
            embeddings[i] = model.wv[str(i)]
        else:
            embeddings[i] = np.random.normal(0, 0.1, embedding_dim)
            
    return torch.tensor(embeddings, dtype=torch.float32)